# EasyOCR experiment

Standalone notebook to try EasyOCR against the real tag photos in `tests/fixtures/`, compare it to the current Tesseract pipeline, and experiment freely. Nothing here touches the app -- it's a sandbox.

**Kernel:** select "Python (igi-ocr)" (the project's conda env -- `easyocr`, `jupyter`, and everything else are already installed there).

**Findings so far** (see below for the code that produced these):
- EasyOCR's raw text recognition is noticeably better than Tesseract's on these photos (REPORT/CVD correct 6/6 vs 3/6, carat content correct 6/6, shape correct 6/6).
- It's slow on CPU: roughly 30-40 seconds per image on this machine. Tesseract takes about a second.
- It pulls in PyTorch + torchvision + scipy + scikit-image -- hundreds of MB, versus Tesseract's tiny footprint. That's a real risk for Streamlit Community Cloud's free tier (build size / memory limits).
- The existing `parsing.py` regexes are tuned for Tesseract's output shape and don't perfectly fit EasyOCR's per-text-box output (color/clarity in particular needs the two boxes merged onto one line, and a couple of small format mismatches like `"1"` read as `"T"`).

## Setup

In [6]:
!pip install easyocr

Defaulting to user installation because normal site-packages is not writeable
  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached opencv_python_headless-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached ninja-1.13.0-py3-none-win_amd64.whl.metadata (5.1 kB)
  Using cached filelock-3.31.1-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached imageio-2.37.4-py3-none-any.whl.metadata (9.6 kB)
  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
Using cached easyocr-1.7.2-py3-none-any.whl (2.9 MB)
   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ------------------------------------- -- 3.9/4.2 MB 19.3 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
!pip install opencv-python


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   - -------------------------------------- 1.8/44.0 MB 16.8 MB/s eta 0:00:03
   ----- ---------------------------------- 5.5/44.0 MB 16.5 MB/s eta 0:00:03
   ------- -------------------------------- 7.9/44.0 MB 14.3 MB/s eta 0:00:03
   -------- ------------------------------- 9.4/44.0 MB 12.5 MB/s eta 0:00:03
   ---------- ----------------------------- 11.5/44.0 MB 11.7 MB/s eta 0:00:03
   ------------ --------------------------- 13.6/44.0 MB 11.5 MB/s eta 0:00:03
   -------------- ------------------------- 15.7/44.0 MB 11.2 MB/s eta 0:00:03
   --------------- ------------------------ 17.3/44.0 MB 11.0 MB/s eta 0:00:03
   ----------------- ---------------------- 19.4/44.0 MB 10.6 MB/s eta 0:00:03
   ------------------- -------------------- 21.5/44.0 MB 10.6 MB/s eta 0:00:03
   --------------------- ------------------ 24.1/44.0 MB 10.8 MB/s


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
!pip install pytesseract pillow


Defaulting to user installation because normal site-packages is not writeable
  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
Using cached pytesseract-0.3.13-py3-none-any.whl (14 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"  # avoids a Windows console crash in EasyOCR's progress bar

import sys
import time
from pathlib import Path

sys.path.insert(0, "..")  # project root, so ocr/imaging/parsing import correctly
# (Jupyter's default working directory is this notebook's own folder)

import cv2
import easyocr
import matplotlib.pyplot as plt

import ocr as tesseract_ocr  # the project's existing Tesseract wrapper (ocr.py)
import parsing

# Tesseract's engine binary isn't on PATH in this kernel -- point pytesseract at
# the winget install directly (see ocr.configure_tesseract's docstring).
os.environ.setdefault("TESSERACT_CMD", r"C:\Program Files\Tesseract-OCR\tesseract.exe")
tesseract_ocr.configure_tesseract()

FIXTURES_DIR = Path("../tests/fixtures")

# Ground truth read directly off each tag photo, for scoring.
CASES = [
    ("sample_tag.jpeg", "809614206", "CVD", "EMERALD", "3.01", "E", "VS1"),
    ("tag_e79422_marquise.jpeg", "791655400", "CVD", "MARQUISE", "2.04", "D", "VVS2"),
    ("tag_c141640_heart.jpeg", "817630270", "CVD", "HEART", "1.00", "F", "VS1"),
    ("tag_e84454_emerald.jpeg", "804633671", "CVD", "EMERALD", "2.97", "E", "VVS1"),
    ("tag_c141641_heart.jpeg", "817634109", "CVD", "HEART", "1.00", "F", "VS1"),
    ("tag_e86943_oval.jpeg", "809609517", "CVD", "OVAL", "1.00", "D", "VVS1"),
]
FIELDS = ["report_type", "shape", "carat", "color", "clarity"]

## Look at one photo

Change `filename` to try a different fixture, or point it at any other image path (e.g. one of your own cropped photos). This and the Tesseract cells below don't touch EasyOCR at all -- no need to wait on the reader load further down.

In [ ]:
filename = "WhatsApp Image 2026-07-20 at 2.44.15 PM.jpeg"
image = cv2.imread(str(FIXTURES_DIR / filename), cv2.IMREAD_COLOR)

plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(filename)
plt.show()

## Run Tesseract on it (raw, no parsing yet)

Runs the project's current Tesseract pipeline on the photo above -- independent of EasyOCR. Tesseract needs the OpenCV preprocessing in `imaging.preprocess` to pick the sparse tag text out from the barcode/QR graphics -- `ocr.run_ocr` then calls it with `--psm 11` (sparse text, no particular order).

In [ ]:
import imaging  # the project's current OpenCV preprocessing, for the Tesseract side

preprocessed = imaging.preprocess(image)

t0 = time.time()
tesseract_text = tesseract_ocr.run_ocr(preprocessed)
print(f"{time.time() - t0:.1f}s")
print(tesseract_text)

## Now parse Tesseract's output

Tesseract already returns continuous multi-line text (unlike EasyOCR's per-box output), so no line-reconstruction step is needed here -- straight into `parsing.parse_fields`.

In [ ]:
tesseract_fields = parsing.parse_fields(tesseract_text)
tesseract_fields

## Load the EasyOCR reader

First run downloads the detection + recognition model weights (a few minutes). After that it's cached locally and loads fast.

In [2]:
reader = easyocr.Reader(["en"], gpu=False, verbose=False)
print("EasyOCR reader ready")

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


EasyOCR reader ready


## Run EasyOCR on it (with timing)

`detail=1` returns `(bbox, text, confidence)` per detected text box -- this is what lets us see roughly where on the tag each piece of text was found, which matters for reconstructing lines (see next cell).

In [4]:
t0 = time.time()
detections = reader.readtext(image, detail=1)
print(f"{time.time() - t0:.1f}s")

for bbox, text, conf in detections:
    print(f"{conf:.2f}  {text!r}")

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


5.8s
0.73  'C141619'
1.00  'REPORT'
0.98  'CVD'
0.27  'Igi CeRT'
1.00  '809614206'
0.68  '8'
1.00  '3.01'
0.50  '3'
0.98  'E'
0.43  'VST'
0.26  'F'
1.00  'EMERALD'
0.12  'E'
0.76  'Cut-VG;'
0.21  "Pal'FX"
0.58  "Svm 'FX"
0.57  "FI 'N"


## Reconstruct text lines from the detected boxes

EasyOCR returns one box per detected text fragment, not one continuous text block like Tesseract. To feed the existing line-based `parsing.parse_fields`, boxes that sit on roughly the same row get grouped into one line (left-to-right). `y_tolerance` controls how close two boxes' vertical centers need to be to count as "the same row" -- tune it if lines are merging wrong or splitting wrong for a given photo.

In [7]:
def group_into_lines(detections, y_tolerance=15):
    items = []
    for bbox, text, conf in detections:
        ys = [p[1] for p in bbox]
        xs = [p[0] for p in bbox]
        items.append((sum(ys) / len(ys), min(xs), text))
    items.sort(key=lambda t: t[0])

    lines, current_line, current_y = [], [], None
    for y, x, text in items:
        if current_y is None or abs(y - current_y) <= y_tolerance:
            current_line.append((x, text))
            current_y = y if current_y is None else (current_y + y) / 2
        else:
            lines.append(" ".join(t for _, t in sorted(current_line)))
            current_line, current_y = [(x, text)], y
    if current_line:
        lines.append(" ".join(t for _, t in sorted(current_line)))
    return "\n".join(lines)


raw_text = group_into_lines(detections)
print(raw_text)

C141619
REPORT
CVD
Igi CeRT 809614206
8 3.01 3
E VST
F E
EMERALD
Cut-VG; Pal'FX Svm 'FX FI 'N


## Parse it with the existing field parser

In [8]:
fields = parsing.parse_fields(raw_text)
fields

{'igi_report_no': '809614206',
 'report_type': 'CVD',
 'shape': 'EMERALD',
 'carat': None,
 'color': None,
 'clarity': None}

## Compare EasyOCR vs Tesseract across all 6 real photos

This is the same scoring approach used earlier in the conversation: for each field, count correct / wrong / missing against the known ground truth. Re-run after changing `group_into_lines` (e.g. its `y_tolerance`) or after editing `parsing.py` to see the effect immediately.

In [ ]:
import imaging  # the project's current OpenCV preprocessing, for the Tesseract side


def score(label, get_raw_text_fn):
    correct = wrong = missing = 0
    total_time = 0.0
    rows = []
    for filename, igi, *expected in CASES:
        img = cv2.imread(str(FIXTURES_DIR / filename), cv2.IMREAD_COLOR)
        t0 = time.time()
        text = get_raw_text_fn(img)
        elapsed = time.time() - t0
        total_time += elapsed
        parsed = parsing.parse_fields(text)
        for field, exp in zip(FIELDS, expected):
            actual = parsed.get(field)
            if actual is None:
                missing += 1
            elif actual == exp:
                correct += 1
            else:
                wrong += 1
        rows.append((filename, elapsed, parsed))
    total = len(CASES) * len(FIELDS)
    print(f"{label}: {correct}/{total} correct, {wrong} wrong, {missing} missing | avg {total_time/len(CASES):.1f}s/image")
    return rows


def tesseract_raw_text(img):
    return tesseract_ocr.run_ocr(imaging.preprocess(img))


def easyocr_raw_text(img):
    return group_into_lines(reader.readtext(img, detail=1))


tesseract_rows = score("Tesseract (current pipeline)", tesseract_raw_text)
easyocr_rows = score("EasyOCR", easyocr_raw_text)

## Per-photo detail

Handy for seeing exactly what each engine parsed out of each photo, side by side.

In [ ]:
for (filename, t_time, t_fields), (_, e_time, e_fields) in zip(tesseract_rows, easyocr_rows):
    print(f"=== {filename} ===")
    print(f"  Tesseract ({t_time:.1f}s): {t_fields}")
    print(f"  EasyOCR   ({e_time:.1f}s): {e_fields}")
    print()

## Try your own photo

Drop an image anywhere under the project and point this at it -- e.g. a fresh crop you want to test.

In [ ]:
my_photo_path = "tests/fixtures/sample_tag.jpeg"  # <-- change this

img = cv2.imread(my_photo_path, cv2.IMREAD_COLOR)
plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

t0 = time.time()
text = easyocr_raw_text(img)
print(f"EasyOCR: {time.time() - t0:.1f}s")
print(text)
print()
print("Parsed:", parsing.parse_fields(text))